In [26]:
import os
import sys
sys.path.append(os.path.abspath(".."))

In [27]:
from app.conversation.prompts import build_context
from app.utils.helper import latest_user_message
from app.utils.helper import is_vague
from app.conversation.recommender import LLMRecommender

In [28]:
import pandas as pd
import numpy as np
from sentence_transformers import SentenceTransformer
from app.retrieval.hybrid import HybridRetriever
from app.retrieval.bm25 import BM25Retriever
from app.retrieval.faiss_retriever import FaissRetriever
from app.conversation.recommender import LLMRecommender

In [29]:
df = pd.read_csv("../data/processed/catalog_processed.csv")
embeddings = np.load(
    "../data/embeddings/catalog_embeddings.npy"
)
embedder = SentenceTransformer(
    "all-MiniLM-L6-v2"
)
bm25 = BM25Retriever(df)
faiss = FaissRetriever(
    df,
    embeddings
)
hybrid = HybridRetriever(
    bm25,
    faiss
)
llm = LLMRecommender()

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [30]:
from app.agent.intent import detect_intent

In [31]:
detect_intent("Recommend assessments for Java developer")

'recommend'

In [32]:
detect_intent("Compare OPQ and OPQ MQ")

'compare'

In [33]:
from app.agent.clarify import needs_clarification

In [34]:
needs_clarification("assessment")

(True, "Could you tell me which role you're hiring for?")

In [35]:
needs_clarification("Java Engineer")

(True, 'Is this for graduates, junior, mid-level, or senior candidates?')

In [36]:
needs_clarification("Graduate Java Engineer")

(False, None)

In [37]:
needs_clarification("Graduate Java Engineer")

(False, None)

In [38]:
needs_clarification("Need leadership assessment")

(True, 'Which role are you hiring for?')

In [39]:
from app.agent.service import AgentService

In [40]:
agent = AgentService(
    hybrid,
    embedder,
    llm
)

In [41]:
conversation = [
    {
        "role": "user",
        "content": "Hello"
    }
]
agent.chat(conversation)

{'reply': 'Hello! How can I help you choose SHL assessments today?',
 'recommendations': []}

In [42]:
conversation = [
    {
        "role":"user",
        "content":"Need assessment"
    }
]

agent.chat(conversation)

{'reply': 'Which role are you hiring for?', 'recommendations': []}

In [43]:
conversation = [
    {
        "role":"user",
        "content":"Graduate Java Engineer"
    }
]

response = agent.chat(conversation)

print(response["reply"])

Here are my explanations for each recommended SHL assessment:

1. Java 8 (New)
Assessment Type: Knowledge & Skills
Job Level: Mid-Professional, Professional Individual Contributor
Duration: 18 minutes
URL: https://www.shl.com/products/product-catalog/view/java-8-new/

This assessment matches the hiring need as it evaluates a candidate's knowledge and skills in Java 8 programming. The focus on mid-professional and professional levels ensures that only experienced engineers are assessed.

Skills evaluated:
* Programming skills in Java 8
* Understanding of Java 8 features and syntax
* Problem-solving abilities

2. Java Design Patterns (New)
Assessment Type: Knowledge & Skills
Job Level: Mid-Professional, Professional Individual Contributor
Duration: 5 minutes
URL: https://www.shl.com/products/product-catalog/view/java-design-patterns-new/

This assessment evaluates a candidate's knowledge and understanding of design patterns in Java. It ensures that candidates can apply design principles 

In [44]:
from app.memory.state import ConversationState

In [45]:
state = ConversationState()
print(state)

ConversationState(role=None, seniority=None, skills=[], assessment_types=[], duration=None, language=None)


In [46]:
state.set_role("Graduate Java Engineer")

state.add_skill("Java")

state.add_skill("Spring")

state.add_assessment_type("Knowledge")

state.set_max_duration(20)

print(state)

ConversationState(role=Graduate Java Engineer, seniority=None, skills=['Java', 'Spring'], assessment_types=['Knowledge'], duration=20, language=None)


In [47]:
from app.memory.extractor import *
from app.memory.state import ConversationState

In [48]:
state = ConversationState()
update_state(
    state,
    "Graduate Java Backend Engineer with Spring Boot SQL under 20 minutes and personality assessment"
)
print(state)

ConversationState(role=Graduate Java Backend Engineer with Spring Boot SQL under 20 minutes and personality assessment, seniority=Graduate, skills=['Java', 'Spring', 'Spring Boot', 'Sql'], assessment_types=['Personality'], duration=20, language=None)


In [49]:
from app.retrieval.filter import *

In [51]:
query = "Graduate Java Engineer"

query_embedding = embedder.encode([query])

results = hybrid.search(
    query=query,
    query_embedding=query_embedding,
    top_k=10
)

len(results)

10

In [52]:
from app.memory.state import ConversationState
from app.retrieval.filter import apply_filters

state = ConversationState()
state.set_max_duration(20)

filtered = apply_filters(results, state)

for r in filtered:
    print(r["name"], "-", r["duration"])

Java 8 (New) - 18 minutes
Java Design Patterns (New) - 5 minutes
Core Java (Advanced Level) (New) - 13 minutes
Graduate Scenarios Narrative Report - nan
Java Frameworks (New) - 17 minutes
Enterprise Java Beans (New) - 4 minutes
Java Web Services (New) - 8 minutes
Core Java (Entry Level) (New) - 13 minutes


In [53]:
from app.memory.state import ConversationState
from app.memory.query_builder import build_search_query

In [54]:
state = ConversationState()
state.set_role("Graduate Java Engineer")
state.add_skill("Java")
state.add_skill("Spring")
state.add_skill("SQL")
print(build_search_query(state))

Graduate Java Engineer Java Spring SQL
